## Annex J. Opensees Python Script for [NLI2] implementation
2nd order inelastic

In [ ]:
import openseespy.opensees as ops
import pandas as pd
import numpy as np
import math

# ---------------------------------------------------------
# 1. INITIALIZATION & SETUP
# ---------------------------------------------------------
ops.wipe()
# 3 Dimensions, 6 Degrees of Freedom per node
ops.model('basic', '-ndm', 3, '-ndf', 6)

print("Starting OpenSeesPy Model Construction...")

# ---------------------------------------------------------
# 2. READ CSV FILES
# ---------------------------------------------------------
try:
    df_props = pd.read_csv('properties.csv')
    df_fix = pd.read_csv('fixities.csv')
    df_loads_slab = pd.read_csv('loads_slab.csv')
    df_loads_sw = pd.read_csv('loads_sw.csv')
except FileNotFoundError as e:
    print(f"Error reading file: {e}")
    print("Ensure properties.csv, fixities.csv, loads_slab.csv, and loads_sw.csv are in the same directory.")
    exit()

# NEW SNIPPET: Check for support displacements
try:
    df_support_disp = pd.read_csv('support_disp.csv')
    print("Found 'support_disp.csv'.")
    # (Applying these as actual settlements requires sp commands in the load pattern, 
    # but we are just catching the file presence for now).
except FileNotFoundError:
    print("No 'support_disp.csv' found. Assuming all support settlements = 0.0")
    df_support_disp = None

# ---------------------------------------------------------
# 3. DEFINE NODES
# ---------------------------------------------------------
# Extract unique nodes from properties.csv (both I and J ends)
nodes_I = df_props[['Node_I', 'Xi', 'Yi', 'Zi']].rename(columns={'Node_I': 'node', 'Xi': 'x', 'Yi': 'y', 'Zi': 'z'})
nodes_J = df_props[['Node_J', 'Xj', 'Yj', 'Zj']].rename(columns={'Node_J': 'node', 'Xj': 'x', 'Yj': 'y', 'Zj': 'z'})
all_nodes = pd.concat([nodes_I, nodes_J]).drop_duplicates(subset=['node']).sort_values('node')

# Add nodes to OpenSees
for _, row in all_nodes.iterrows():
    ops.node(int(row['node']), float(row['x']), float(row['y']), float(row['z']))

print(f"Defined {len(all_nodes)} nodes.")

# ---------------------------------------------------------
# 4. DEFINE SUPPORTS (FIXITIES)
# ---------------------------------------------------------
# Clean string formats, apply fixities
supported_nodes = []
for _, row in df_fix.iterrows():
    node_tag = int(row['node'])
    
    # Helper to check if string contains 'fixed'
    def is_fixed(val):
        return 1 if str(val).strip().lower() == 'fixed' else 0

    fix_x = is_fixed(row['x'])
    fix_y = is_fixed(row['y'])
    fix_z = is_fixed(row['z'])
    fix_rx = is_fixed(row['x-rot'])
    fix_ry = is_fixed(row['y-rot'])
    fix_rz = is_fixed(row['z-rot'])
    
    ops.fix(node_tag, fix_x, fix_y, fix_z, fix_rx, fix_ry, fix_rz)
    
    # Keep track for outputting reactions later
    if sum([fix_x, fix_y, fix_z, fix_rx, fix_ry, fix_rz]) > 0:
        supported_nodes.append(node_tag)

print(f"Defined {len(supported_nodes)} supported nodes.")

# ---------------------------------------------------------
# 5. DEFINE INELASTIC ELEMENTS & FIBER PROPERTIES
# ---------------------------------------------------------
print("Defining Inelastic Fiber Sections...")

# A. Define Global Uniaxial Materials (Units: kN, mm -> MPa / 1000 = kN/mm2)
mat_tag_conc = 1
# Concrete01: 30 MPa strength
fpc = -0.030      # Peak compressive strength
epsc0 = -0.002    # Strain at peak strength
fpcu = -0.006     # Crushing residual strength
epsU = -0.005     # Strain at ultimate crushing
ops.uniaxialMaterial('Concrete01', mat_tag_conc, fpc, epsc0, fpcu, epsU)

mat_tag_steel = 2
# Steel01: Grade 60 Rebar (414 MPa)
Fy = 0.414        # Yield strength
E_steel = 200.0   # Modulus of elasticity (200,000 MPa = 200 kN/mm2)
b_steel = 0.01    # Strain-hardening ratio
ops.uniaxialMaterial('Steel01', mat_tag_steel, Fy, E_steel, b_steel)

transf_tag = 1

for _, row in df_props.iterrows():
    ele_tag = int(row['Member_ID'])
    node_i = int(row['Node_I'])
    node_j = int(row['Node_J'])
    
    A = float(row['Area (mm2)'])
    E = float(row['E (GPa)']) 
    v = float(row['v'])
    J = float(row['J (mm4)'])
    Iy = float(row['Iy (mm4)'])
    Iz = float(row['Iz (mm4)'])
    G = E / (2.0 * (1.0 + v))
    
    # 1. Reverse-engineer physical dimensions from A and Iz (assuming rectangular)
    # Area = b*h, Iz = (b*h^3)/12  -->  h = sqrt(12*Iz / Area)
    depth = math.sqrt((12.0 * Iz) / A)
    width = A / depth
    
    # 2. Geometric Transformation Vector Logic
    dx = float(row['Xj']) - float(row['Xi'])
    dy = float(row['Yj']) - float(row['Yi'])
    dz = float(row['Zj']) - float(row['Zi'])
    L = math.sqrt(dx**2 + dy**2 + dz**2)
    
    if abs(dx/L) < 1e-6 and abs(dz/L) < 1e-6:
        vecxz = [0.0, 0.0, 1.0] 
    elif abs(dz/L) > 0.999:
        vecxz = [1.0, 0.0, 0.0]
    else:
        vecxz = [0.0, 0.0, 1.0] 
        
    ops.geomTransf('PDelta', transf_tag, *vecxz)
    
    # 3. Build Fiber Section (Tag matches ele_tag for unique mapping)
    sec_tag = ele_tag
    
    # --- MODERN OPENSEES SYNTAX: Apply Torsion (G * J) directly here ---
    ops.section('Fiber', sec_tag, '-GJ', G * J)
    
    # Concrete Patch
    ops.patch('rect', mat_tag_conc, 15, 15, -depth/2, -width/2, depth/2, width/2)
    
    # Steel Layers (approx 1.5% of gross area, divided into top and bottom layers)
    As_total = 0.015 * A
    As_layer = As_total / 2.0
    cover = min(40.0, depth/4.0) 
    
    y_top = (depth / 2.0) - cover
    y_bot = -(depth / 2.0) + cover
    z_edge = (width / 2.0) - cover
    
    ops.layer('straight', mat_tag_steel, 3, As_layer/3.0, y_top, -z_edge, y_top, z_edge)
    ops.layer('straight', mat_tag_steel, 3, As_layer/3.0, y_bot, -z_edge, y_bot, z_edge)
    
    # 4. (The Aggregator block has been entirely deleted because it is no longer needed)
    
    # 5. Define Beam Integration and Inelastic Element
    integration_tag = ele_tag + 30000
    
    # Pass 'sec_tag' directly into the integration command instead of an aggregator
    ops.beamIntegration('Lobatto', integration_tag, sec_tag, 5) 
    
    # Create the forceBeamColumn
    ops.element('forceBeamColumn', ele_tag, node_i, node_j, transf_tag, integration_tag)
    
    transf_tag += 1

print(f"Defined {len(df_props)} inelastic elements.")

# ---------------------------------------------------------
# 6. DEFINE LOADS & SETTLEMENTS (REVERTED TO NODAL LOADS)
# ---------------------------------------------------------
ops.timeSeries('Linear', 1)
ops.pattern('Plain', 1, 1)

# --- PART A: Apply Nodal Forces ---
# Combine your slab and self-weight dataframes
df_all_loads = pd.concat([df_loads_slab, df_loads_sw], ignore_index=True)

# Group by 'node' (not Member_ID!) to sum up multiple loads on the same joint
df_grouped_loads = df_all_loads.groupby('node', as_index=False).sum()

for _, row in df_grouped_loads.iterrows():
    node_tag = int(row['node'])
    
    # Extract forces and moments from your CSV columns
    fx = float(row['Fx'])
    fy = float(row['Fy'])
    fz = float(row['Fz'])
    mx = float(row['Mx'])
    my = float(row['My'])
    mz = float(row['Mz'])
    
    # Apply them directly to the node
    ops.load(node_tag, fx, fy, fz, mx, my, mz)

print("Applied nodal loads successfully.")

# --- PART B: Apply Support Displacements (Settlements) ---
if df_support_disp is not None:
    # If coordinates are scaled to mm, ensure settlements are scaled to mm too if needed!
    UNIT_SCALE = 1000.0  # Change to 1.0 if your support_disp.csv is ALREADY in mm
    
    for _, row in df_support_disp.iterrows():
        node_tag = int(row['node'])
        
        if float(row['dx']) != 0.0: ops.sp(node_tag, 1, float(row['dx']) * UNIT_SCALE)
        if float(row['dy']) != 0.0: ops.sp(node_tag, 2, float(row['dy']) * UNIT_SCALE)
        if float(row['dz']) != 0.0: ops.sp(node_tag, 3, float(row['dz']) * UNIT_SCALE)
        if float(row['rx']) != 0.0: ops.sp(node_tag, 4, float(row['rx']))
        if float(row['ry']) != 0.0: ops.sp(node_tag, 5, float(row['ry']))
        if float(row['rz']) != 0.0: ops.sp(node_tag, 6, float(row['rz']))
        
    print("Applied prescribed support displacements from CSV.")

# ---------------------------------------------------------
# 7. ANALYSIS SETUP
# ---------------------------------------------------------
# # First-order linear elastic analysis parameters
# ops.system('BandGeneral')
# ops.numberer('RCM')
# ops.constraints('Transformation')
# ops.integrator('LoadControl', 1.0)
# ops.algorithm('Linear')
# ops.analysis('Static')

# # Run analysis
# print("Running Linear Analysis...")
# ok = ops.analyze(1)

# if ok == 0:
#     print("Analysis completed successfully!")
# else:
#     print("Analysis failed.")
#     exit()

# ---------------------------------------------------------
# 7. 2ND-ORDER INELASTIC ANALYSIS SETUP
# ---------------------------------------------------------
ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Transformation')

# 1. Displacement Convergence test
ops.test('NormDispIncr', 1.0e-5, 100) 

# 2. Use 'NewtonLineSearch' instead of standard 'Newton'
# Line Search introduces a damping factor to help the solver find equilibrium 
# when concrete elements crack or yield under P-Delta sways.
ops.algorithm('NewtonLineSearch')

# 3. Apply the load in smaller steps (e.g., 20 steps of 5% load)
# This provides a smoother numerical path for the 2nd-order geometry to update.
ops.integrator('LoadControl', 0.05)
ops.analysis('Static')

# Run analysis for 20 steps to reach 100% load (20 * 0.05 = 1.0)
print("Running 2nd-Order Inelastic Analysis (20 steps)...")
ok = ops.analyze(20)

if ok == 0:
    print("2nd-Order Inelastic Analysis converged successfully!")
else:
    print("Analysis failed to converge. The structure may have buckled or reached its ultimate load capacity.")
    exit()

# =========================================================================
# 8. EXTRACT RESULTS (TRANSFORMED DIRECTLY TO YOUR EXCEL AXES CONVENTION)
# =========================================================================
import numpy as np
import pandas as pd

# --- TOGGLE SWITCH FOR DEBUG LEDGER ---
ENABLE_DEBUG_LEDGER = True  # Set to False to turn off console printing

ops.reactions()  # Populate node reaction values

# --- DEFINE LOOKUP: Combine slab & self-weight dataframes ---
df_all_loads = pd.concat([df_loads_slab, df_loads_sw], ignore_index=True)
df_grouped_loads = df_all_loads.groupby('node', as_index=False).sum()
nodal_load_lookup = df_grouped_loads.set_index('node').to_dict(orient='index')

all_nodes_data = [] 
support_reactions_data = []
local_forces_data = [] 

# 1. Get Deformations & Global Reactions
for node_tag in all_nodes['node'].astype(int):
    disps = ops.nodeDisp(node_tag)
    all_nodes_data.append({
        'Node': node_tag,
        'Disp_x (mm)': disps[0], 'Disp_y (mm)': disps[1], 'Disp_z (mm)': disps[2],
        'Rot_x (rad)':  disps[3], 'Rot_y (rad)':  disps[4], 'Rot_z (rad)':  disps[5]
    })
    
    if node_tag in supported_nodes:
        rxns = ops.nodeReaction(node_tag)
        support_reactions_data.append({
            'Node': node_tag,
            'Rxn_Fx (kN)': rxns[0], 'Rxn_Fy (kN)': rxns[1], 'Rxn_Fz (kN)': rxns[2],
            'Rxn_Mx (kN-mm)': rxns[3], 'Rxn_My (kN-mm)': rxns[4], 'Rxn_Mz (kN-mm)': rxns[5]
        })

# 2. Get Member Forces using Excel's Exact Rotation Matrix Logic
for _, row in df_props.iterrows():
    ele_tag = int(row['Member_ID'])
    node_i = int(row['Node_I'])
    node_j = int(row['Node_J'])
    
    # Native OpenSees Local Response
    k_u_local = ops.eleResponse(ele_tag, 'localForce')
    
    if k_u_local:
        coord_i = np.array(ops.nodeCoord(node_i))
        coord_j = np.array(ops.nodeCoord(node_j))
        
        # Calculate standard directional unit vector (Local X)
        v_x = coord_j - coord_i
        L = np.linalg.norm(v_x)
        lx, mx, nx = v_x / L
        
        # A. BUILD OPENSEES INTERNAL TRANSFORMATION MATRIX (To reverse-engineer back to global)
        # Tracking the original baseline logic vectors used in Section 5
        if abs(lx) < 1e-6 and abs(nx) < 1e-6:
            vecxz_os = np.array([0.0, 0.0, 1.0])
        elif abs(nx) > 0.999:
            vecxz_os = np.array([1.0, 0.0, 0.0])
        else:
            vecxz_os = np.array([0.0, 0.0, 1.0])
            
        v_y_os = np.cross(vecxz_os, [lx, mx, nx])
        e_y_os = v_y_os / np.linalg.norm(v_y_os)
        e_z_os = np.cross([lx, mx, nx], e_y_os)
        R_os = np.vstack([[lx, mx, nx], e_y_os, e_z_os]) # 3x3 OpenSees Matrix
        
        # B. BUILD YOUR EXACT EXCEL ROTATION MATRIX
        D = math.sqrt(lx**2 + nx**2) # Enforcing the mathematical SQRT behavior of your D parameter
        if D == 0:
            ly, my, ny = -mx, 0.0, 0.0
            lz, mz, nz = 0.0, 0.0, 1.0
        else:
            ly, my, ny = -(lx * mx) / D, D, -(mx * nx) / D
            lz, mz, nz = -nx / D, 0.0, lx / D
            
        R_excel = np.array([[lx, mx, nx], [ly, my, ny], [lz, mz, nz]]) # 3x3 Excel Matrix

        # C. SEPARATE FORCES & MOMENTS FOR NODE I AND J
        f_os_i = np.array(k_u_local[0:3])
        m_os_i = np.array(k_u_local[3:6])
        f_os_j = np.array(k_u_local[6:9])
        m_os_j = np.array(k_u_local[9:12])
        
        # D. ROTATE OPENSEES NATIVE FORCES BACK TO GLOBAL COORDS
        f_global_i = R_os.T @ f_os_i
        m_global_i = R_os.T @ m_os_i
        f_global_j = R_os.T @ f_os_j
        m_global_j = R_os.T @ m_os_j
        
        # E. FETCH GLOBAL FIXED END FORCES FROM CSV
        fe_i = nodal_load_lookup.get(node_i, {'Fx':0., 'Fy':0., 'Fz':0., 'Mx':0., 'My':0., 'Mz':0.})
        fe_j = nodal_load_lookup.get(node_j, {'Fx':0., 'Fy':0., 'Fz':0., 'Mx':0., 'My':0., 'Mz':0.})
        
        P_g_fixed_f_i = np.array([fe_i['Fx'], fe_i['Fy'], fe_i['Fz']])
        P_g_fixed_m_i = np.array([fe_i['Mx'], fe_i['My'], fe_i['Mz']])
        P_g_fixed_f_j = np.array([fe_j['Fx'], fe_j['Fy'], fe_j['Fz']])
        P_g_fixed_m_j = np.array([fe_j['Mx'], fe_j['My'], fe_j['Mz']])
        
        # F. APPLY COMBINED SUBTRACTION FORMULA IN GLOBAL COORDINATES
        f_g_final_i = f_global_i - P_g_fixed_f_i
        m_g_final_i = m_global_i - P_g_fixed_m_i
        f_g_final_j = f_global_j - P_g_fixed_f_j
        m_g_final_j = m_global_j - P_g_fixed_m_j
        
        # G. ROTATE THE TOTAL FINAL FORCES DIRECTLY INTO YOUR EXCEL LOCAL COORDS
        f_excel_i = R_excel @ f_g_final_i
        m_excel_i = R_excel @ m_g_final_i
        f_excel_j = R_excel @ f_g_final_j
        m_excel_j = R_excel @ m_g_final_j
        
        local_forces_data.append({
            'Member_ID': ele_tag, 'Node_I': node_i, 'Node_J': node_j,
            'Fx_I (kN)': f_excel_i[0], 'Fy_I (kN)': f_excel_i[1], 'Fz_I (kN)': f_excel_i[2],
            'Mx_I (kN-mm)': m_excel_i[0], 'My_I (kN-mm)': m_excel_i[1], 'Mz_I (kN-mm)': m_excel_i[2],
            'Fx_J (kN)': f_excel_j[0], 'Fy_J (kN)': f_excel_j[1], 'Fz_J (kN)': f_excel_j[2],
            'Mx_J (kN-mm)': m_excel_j[0], 'My_J (kN-mm)': m_excel_j[1], 'Mz_J (kN-mm)': m_excel_j[2]
        })
        
        # --- PRINT OUT LEDGER MAPPED EXACTLY TO YOUR EXCEL CONVENTION ---
        if ENABLE_DEBUG_LEDGER:
            # 1. Transform all individual components to Excel local space for column consistency
            f_os_excel_i = R_excel @ (R_os.T @ f_os_i)
            m_os_excel_i = R_excel @ (R_os.T @ m_os_i)
            f_os_excel_j = R_excel @ (R_os.T @ f_os_j)
            m_os_excel_j = R_excel @ (R_os.T @ m_os_j)
            
            f_fixed_excel_i = R_excel @ P_g_fixed_f_i
            m_fixed_excel_i = R_excel @ P_g_fixed_m_i
            f_fixed_excel_j = R_excel @ P_g_fixed_f_j
            m_fixed_excel_j = R_excel @ P_g_fixed_m_j

            # 2. Package into 6DOF arrays for clean iterative printing
            os_ku_i  = np.concatenate([f_os_excel_i, m_os_excel_i])
            csv_g_i  = np.concatenate([P_g_fixed_f_i, P_g_fixed_m_i])
            fixed_l_i = np.concatenate([f_fixed_excel_i, m_fixed_excel_i])
            final_c_i = np.concatenate([f_excel_i, m_excel_i])

            os_ku_j  = np.concatenate([f_os_excel_j, m_os_excel_j])
            csv_g_j  = np.concatenate([P_g_fixed_f_j, P_g_fixed_m_j])
            fixed_l_j = np.concatenate([f_fixed_excel_j, m_fixed_excel_j])
            final_c_j = np.concatenate([f_excel_j, m_excel_j])

            # 3. Print out to your exact console layout specifications
            print(f"\n" + "="*95)
            print(f" DEBUG LEDGER FOR MEMBER {ele_tag} (Node {node_i} -> Node {node_j}) | Length: {L:.2f} mm")
            print(f" Transf Vector used: vecxz = {vecxz_os}")
            print(f"-"*95)
            print(" [r]=")
            print(f" {R_excel[0,0]:8.4f} {R_excel[0,1]:8.4f} {R_excel[0,2]:8.4f}   (lx, mx, nx)")
            print(f" {R_excel[1,0]:8.4f} {R_excel[1,1]:8.4f} {R_excel[1,2]:8.4f}   (ly, my, ny)")
            print(f" {R_excel[2,0]:8.4f} {R_excel[2,1]:8.4f} {R_excel[2,2]:8.4f}   (lz, mz, nz)")
            print(f"="*95)
            
            headers = f"{'Component':<12} | {'OpenSees [k][u]':>15} | {'Global CSV':>15} | {'Local Fixed (R*P)':>18} | {'Final Combined':>15}"
            print(headers)
            print("-"*95)
            
            labels = ['Fx (Axial)', 'Fy (V-Shear)', 'Fz (O-Shear)', 'Mx (Torsion)', 'My (O-Bending)', 'Mz (P-Bending)']
            
            print(f"--- NODE I (Start Node: {node_i}) ---")
            for i in range(6):
                print(f"{labels[i]:<12} | {os_ku_i[i]:15.4f} | {csv_g_i[i]:15.4f} | {fixed_l_i[i]:18.4f} | {final_c_i[i]:15.4f}")
                
            print(f"\n--- NODE J (End Node: {node_j}) ---")
            for i in range(6):
                print(f"{labels[i]:<12} | {os_ku_j[i]:15.4f} | {csv_g_j[i]:15.4f} | {fixed_l_j[i]:18.4f} | {final_c_j[i]:15.4f}")
            print("="*95 + "\n")

# Export to CSV
df_disp = pd.DataFrame(all_nodes_data)
df_rxns = pd.DataFrame(support_reactions_data)
df_forces = pd.DataFrame(local_forces_data) 

df_disp.to_csv('results_deformations_opy4.csv', index=False)
df_rxns.to_csv('results_reactions_opy4.csv', index=False)
df_forces.to_csv('results_local_forces_opy4.csv', index=False) 

print("Results converted and exported to match your Excel coordinate sheet layout perfectly.")




In [ ]:
def format_engineering(val, precision=3):
    """
    Formats a numeric value into true engineering notation.
    Numerically zero values output a clean, uniform string.
    """
    try:
        val_float = float(val)
    except (ValueError, TypeError):
        return str(val)
    if abs(val_float) < 1e-9:
        return "0"
        
    sign = "-" if val_float < 0 else ""
    val_abs = abs(val_float)
    
    exp = int(math.floor(math.log10(val_abs) / 3.0) * 3)
    mantissa = val_abs / (10**exp)
    
    if round(mantissa, precision) >= 1000.0:
        mantissa /= 1000.0
        exp += 3
        
    return f"{sign}{mantissa:.{precision}f}e{exp:+03d}"

In [ ]:
# target_columns = [  'Disp_x (mm)',
#  'Disp_y (mm)',
#  'Disp_z (mm)',
#  'Rot_x (rad)',
#  'Rot_y (rad)',
#  'Rot_z (rad)']

# # Apply to just those columns
# df_disp[target_columns] = df_disp[target_columns].map(format_engineering)

# df_disp

In [ ]:
df_disp.to_clipboard()

In [ ]:
df_rxns.to_clipboard()

In [ ]:
df_forces.to_clipboard()

In [ ]:
df_forces

In [ ]:
# ---------------------------------------------------------
# EXTRACTION & VERIFICATION LOOP
# ---------------------------------------------------------
for _, row in df_props.iterrows():
    ele_tag = int(row['Member_ID'])
    
    # 'localForce' returns a list of 12 values: 
    # [Fx_i, Fy_i, Fz_i, Mx_i, My_i, Mz_i,  Fx_j, Fy_j, Fz_j, Mx_j, My_j, Mz_j]
    forces = ops.eleResponse(ele_tag, 'localForce')
    
    print(f"================ MEMBER {ele_tag} ================")
    print(f"--- NODE I (Start Node {int(row['Node_I'])}) ---")
    print(f"Axial Force (Fx):             {forces[0]:.4f}")
    print(f"Vertical Shear (Fy):          {forces[1]:.4f}")
    print(f"Out-of-Plane Moment (My):     {forces[4]:.4f}")
    print(f"PRIMARY BENDING MOMENT (Mz):  {forces[5]:.4f}  <-- COMPARE TO MCGUIRE")
    
    print(f"--- NODE J (End Node {int(row['Node_J'])}) ---")
    print(f"Axial Force (Fx):             {forces[6]:.4f}")
    print(f"Vertical Shear (Fy):          {forces[7]:.4f}")
    print(f"Out-of-Plane Moment (My):     {forces[10]:.4f}")
    print(f"PRIMARY BENDING MOMENT (Mz):  {forces[11]:.4f} <-- COMPARE TO MCGUIRE\n")

In [ ]:
import numpy as np

# ---------------------------------------------------------
# COMPREHENSIVE MATRIX EXTRACTION & VERIFICATION LOOP
# ---------------------------------------------------------
for _, row in df_props.iterrows():
    ele_tag = int(row['Member_ID'])
    node_i = int(row['Node_I'])
    node_j = int(row['Node_J'])
    
    # 1. Native OpenSees Local Response: P_os = [k_local]{u_local}
    k_u_local = ops.eleResponse(ele_tag, 'localForce')
    
    if k_u_local:
        # 2. Get Node Coordinates directly from OpenSees to track orientation
        coord_i = np.array(ops.nodeCoord(node_i))
        coord_j = np.array(ops.nodeCoord(node_j))
        
        # 3. Calculate Local x-axis unit vector
        v_x = coord_j - coord_i
        L = np.linalg.norm(v_x)
        e_x = v_x / L
        
        # 4. Define vecxz explicitly (MUST match your geomTransf exactly!)
        if abs(e_x[1]) > 0.99: 
            vecxz = np.array([1.0, 0.0, 0.0])  # Update to match your column geomTransf
        else:
            # Update this to match your specific horizontal beam geomTransf
            vecxz = np.array([-1.0, 0.0, 0.0]) # Note: Based on your expected [r] output, this yields the correct rotation for e_x = [0, 0, 1]
            
        # 5. EXACT OpenSees C++ Vector Mechanics (Y first, then Z)
        v_y = np.cross(vecxz, e_x)
        e_y = v_y / np.linalg.norm(v_y)
        
        v_z = np.cross(e_x, e_y)
        e_z = v_z / np.linalg.norm(v_z)
        
        R_3x3 = np.vstack([e_x, e_y, e_z])
        
        # 6. Build 6x6 Node Rotation Matrix
        R_6x6 = np.zeros((6, 6))
        R_6x6[0:3, 0:3] = R_3x3
        R_6x6[3:6, 3:6] = R_3x3
        
        # 7. Fetch Global Fixed End Forces from CSV data lookup
        fe_i = nodal_load_lookup.get(node_i, {'Fx':0., 'Fy':0., 'Fz':0., 'Mx':0., 'My':0., 'Mz':0.})
        fe_j = nodal_load_lookup.get(node_j, {'Fx':0., 'Fy':0., 'Fz':0., 'Mx':0., 'My':0., 'Mz':0.})
        
        P_global_fixed_i = np.array([fe_i['Fx'], fe_i['Fy'], fe_i['Fz'], fe_i['Mx'], fe_i['My'], fe_i['Mz']])
        P_global_fixed_j = np.array([fe_j['Fx'], fe_j['Fy'], fe_j['Fz'], fe_j['Mx'], fe_j['My'], fe_j['Mz']])
        
        # 8. Calculate Local Fixed End Forces: {P_local_fixed} = [R] * {P_global_fixed}
        P_local_fixed_i = R_6x6 @ P_global_fixed_i
        P_local_fixed_j = R_6x6 @ P_global_fixed_j
        
        # 9. Apply Final Subtraction: {P_final} = [k]{u} - {P_local_fixed}
        P_final_i = np.array(k_u_local[0:6]) - P_local_fixed_i
        P_final_j = np.array(k_u_local[6:12]) - P_local_fixed_j
        
        # --- PRINT OUT ANALYSIS LEDGER ---
        print(f"\n" + "="*95)
        print(f" DEBUG LEDGER FOR MEMBER {ele_tag} (Node {node_i} -> Node {node_j}) | Length: {L:.2f} mm")
        print(f" Transf Vector used: vecxz = {vecxz}")
        print(f"-"*95)
        
        # Print 3x3 Rotation Matrix (Direction Cosines)
        print(" [r]=")
        print(f" {R_3x3[0,0]:8.4f} {R_3x3[0,1]:8.4f} {R_3x3[0,2]:8.4f}   (lx, mx, nx)")
        print(f" {R_3x3[1,0]:8.4f} {R_3x3[1,1]:8.4f} {R_3x3[1,2]:8.4f}   (ly, my, ny)")
        print(f" {R_3x3[2,0]:8.4f} {R_3x3[2,1]:8.4f} {R_3x3[2,2]:8.4f}   (lz, mz, nz)")
        print(f"="*95)
        
        headers = f"{'Component':<12} | {'OpenSees [k][u]':>15} | {'Global CSV':>15} | {'Local Fixed (R*P)':>18} | {'Final Combined':>15}"
        print(headers)
        print("-"*95)
        
        labels = ['Fx (Axial)', 'Fy (V-Shear)', 'Fz (O-Shear)', 'Mx (Torsion)', 'My (O-Bending)', 'Mz (P-Bending)']
        
        # Print Node I Breakdown
        print(f"--- NODE I (Start Node: {node_i}) ---")
        for i in range(6):
            print(f"{labels[i]:<12} | {k_u_local[i]:15.4f} | {P_global_fixed_i[i]:15.4f} | {P_local_fixed_i[i]:18.4f} | {P_final_i[i]:15.4f}")
            
        # Print Node J Breakdown
        print(f"\n--- NODE J (End Node: {node_j}) ---")
        for i in range(6):
            print(f"{labels[i]:<12} | {k_u_local[i+6]:15.4f} | {P_global_fixed_j[i]:15.4f} | {P_local_fixed_j[i]:18.4f} | {P_final_j[i]:15.4f}")
        print("="*95 + "\n")

In [2]:
import os
import nbformat
from nbconvert import HTMLExporter
from weasyprint import HTML, CSS

def export_notebook_to_annex_e(notebook_path, output_pdf_path="Annex_E_Report.pdf"):
    print(f"Reading notebook: {notebook_path}...")
    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook_content = nbformat.read(f, as_version=4)
        
    # 1. Initialize the HTML Exporter from nbconvert
    html_exporter = HTMLExporter()
    # Optional: exclude input or output prompts if you want a cleaner look
    # html_exporter.exclude_input_prompt = True
    # html_exporter.exclude_output_prompt = True
    
    # 2. Convert the notebook to standard HTML string
    (body, resources) = html_exporter.from_notebook_node(notebook_content)
    
    # 3. Define the strict CSS Paged Media rules for Annex I
    # This automatically builds the header, footer page counts, and handles line wrapping
    custom_css = """
    @page {
        size: A4 portrait;
        margin: 20mm 15mm 20mm 15mm;
        
        @top-right {
            content: "Annex J. Opensees Python Script for [NLI2] implementation";
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
            font-weight: bold;
        }
        
        @bottom-right {
            /* WeasyPrint dynamically computes page and pages variables natively */
            content: "Page J." counter(page) " of J." counter(pages);
            font-family: 'Courier New', Courier, monospace;
            font-size: 8.5pt;
            color: #333333;
        }
    }
    
    /* Global formatting fixes for high-quality printing */
    body {
        font-family: 'Courier New', Courier, monospace !important;
        font-size: 9pt !important;
    }
    
    /* Force long lines of code and text inputs to wrap cleanly instead of clipping horizontally */
    pre, code, .highlight, .input_area, .output_text pre {
        white-space: pre-wrap !important;
        word-wrap: break-word !important;
        word-break: break-all !important;
    }
    
    /* Ensure markdown headers inside the notebook look clean and intentional */
    h1, h2, h3, h4 {
        font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif !important;
        page-break-after: avoid !important;
    }
    
    /* Avoid cutting an atomic code cell cleanly across two pages if possible */
    .cell {
        page-break-inside: auto !important;
    }
    .input_area, .output_wrapper {
        page-break-inside: avoid !important;
    }
    """
    
    # 4. Inject our custom CSS block directly into the HTML body header
    styled_html = body.replace("</head>", f"<style>{custom_css}</style></head>")
    
    print("Compiling styled HTML into final PDF...")
    # 5. Compile to PDF via WeasyPrint
    HTML(string=styled_html).write_pdf(output_pdf_path)
    print(f"Successfully generated: {output_pdf_path}")

# ==============================================================================
# EXECUTION
# ==============================================================================
if __name__ == "__main__":
    # Replace with your actual notebook file name
    target_notebook = "Abcede_CE201_opensees_v4.ipynb" 
    
    if os.path.exists(target_notebook):
        export_notebook_to_annex_e(target_notebook, "Annex_J_opy_NLI2.pdf")
    else:
        print(f"Error: Could not find notebook file '{target_notebook}' in the current directory.")

Reading notebook: Abcede_CE201_opensees_v4.ipynb...
Compiling styled HTML into final PDF...
Successfully generated: Annex_J_opy_NLI2.pdf
